# Train the autoregressive model on Colab (boxes or cylinders)

One-click GPU run for the **A3 / Stage 2** autoregressive CAD-as-language model.
Same recipe (d=256, 5 layers, 100 epochs, ~10k samples) works for either
content type; switch via `DATA_KIND` in cell 3.

**Setup:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

What this does (~15–25 min on a T4):
1. Clone the repo, install deps, verify GPU.
2. Generate 10k training + 400 held-out scenes for the chosen content (~4 min).
3. Train for 100 epochs with d_model=256, 5 decoder layers (~10–18 min on T4).
4. Score under F1 + exact-match; compare against the corresponding reference.
5. Beam-3 ablation on the same checkpoint.
6. Render before/after wireframes.
7. (Optional) Save the checkpoint to Google Drive.

**Reference points** (boxes is the Stage 1 ship config; cylinders is Unit E6):

| Held-out content | F1 | Exact-match |
|-----|-----|-----|
| classical baseline (boxes) | 0.000 | 0.000 |
| AR big preset on **boxes** (Stage 1 ship) | **0.980** | **0.932** |
| AR big preset on **cylinders** (this run target) | — | **>= 0.80** |


## 1. Clone the repo

Edit `REPO_URL` below to point at your fork/push of the project.


In [ ]:
REPO_URL = 'https://github.com/Nollis/lines.git'
BRANCH = 'main'
REPO_DIR = '/content/lines'

import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
os.chdir(REPO_DIR)
print('repo at', os.getcwd())
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())


## 2. Install deps + verify GPU


In [ ]:
!pip install -q -e . matplotlib
import torch
print('torch', torch.__version__, '|  CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('NOTE: no GPU. Runtime > Change runtime type > T4 GPU, then Run all again.')


## 3. Pick the content type

Switch this once at the top of the notebook; data dirs, training output dir,
and eval references all derive from it. The same recipe ships either kind.


In [ ]:
DATA_KIND = 'mixed'   # 'boxes' | 'cylinders' | 'mixed' (the E7 Stage-2 ship test)

# Optional: skip training entirely and load a previously-trained checkpoint
# from Google Drive. Set DATA_KIND to match the checkpoint's content type, then
# point this at the file on Drive (e.g. '/content/drive/MyDrive/ar_mixed_64.pt').
# Leave as None to train fresh.
LOAD_FROM_DRIVE = None

DATASETS = {
    'boxes': {
        'script':       'scripts/build_box_data.py',
        'train_dir':    'data/train64_box_big',
        'test_dir':     'data/test64_box',
        'out_dir':      'checkpoints/ar_box64_gpu_big',
        'extra_args':   [],
        'reference':    'boxes -- Stage 1 ship: F1=0.980, exact=0.932',
    },
    'cylinders': {
        'script':       'scripts/build_cylinder_data.py',
        'train_dir':    'data/train64_cyl',
        'test_dir':     'data/test64_cyl',
        'out_dir':      'checkpoints/ar_cyl64',
        'extra_args':   [],
        'reference':    'cylinders -- Stage 2 Unit E6 ship: F1=0.995, exact=0.988',
    },
    'mixed': {
        'script':       'scripts/build_mixed_3d_data.py',
        'train_dir':    'data/train64_mixed_3d',
        # eval uses the PURE held-out splits separately (Gate 4: both have to pass)
        'test_dir':     'data/test64_box',
        'out_dir':      'checkpoints/ar_mixed_3d',
        'extra_args':   ['--fraction-cyl', '0.5'],
        'reference':    'mixed boxes+cylinders -- Stage 2 Unit E7 ship; Gate 4: boxes >=0.88 AND cyl >=0.80',
    },
}
_cfg = DATASETS[DATA_KIND]
BUILD_SCRIPT  = _cfg['script']
BUILD_EXTRA   = _cfg['extra_args']
TRAIN_DIR     = _cfg['train_dir']
TEST_DIR      = _cfg['test_dir']
OUT_DIR       = _cfg['out_dir']
print(f'DATA_KIND = {DATA_KIND!r}')
print(f'  train:  {TRAIN_DIR}')
print(f'  test:   {TEST_DIR}  (mixed also evaluates on data/test64_cyl)')
print(f'  out:    {OUT_DIR}')
print(f'  ref:    {_cfg["reference"]}')


## 4. Generate data (deterministic seeds -> identical content every time)


In [ ]:
# 10k train + 400 held-out per pure split, deterministic seeds. ~4 min on Colab CPU.
# Disjoint dirs per DATA_KIND so previous content on disk doesn't shadow this.
from pathlib import Path
import subprocess

tasks = [
    (TRAIN_DIR, 10000, 0, BUILD_SCRIPT, BUILD_EXTRA),
    # always build both pure test splits when DATA_KIND='mixed' so Gate 4 can
    # measure each content type separately. For 'boxes'/'cylinders' it's a
    # no-op for the other split (skipped if present, generated otherwise).
    ('data/test64_box', 400, 900_000, 'scripts/build_box_data.py', ['--no-randomize']),
]
if DATA_KIND in ('cylinders', 'mixed'):
    tasks.append(('data/test64_cyl', 400, 900_000, 'scripts/build_cylinder_data.py', ['--no-randomize']))

for split, n, seed, script, extra in tasks:
    if (Path(split) / 'manifest.json').exists():
        print(f'{split}: already on disk, skipping')
        continue
    print(f'generating {split} ({n} samples) via {script}...')
    subprocess.check_call(['python', script, '--out', split, '--n', str(n),
                            '--seed0', str(seed), '--canvas-side', '64'] + extra)


## 5. Train (BIG preset: 100 epochs, ~10–18 min on T4)

Calls the training function directly so the loss log streams into this cell.
Periodic checkpointing every 5 epochs (`OUT_DIR/model.pt`) makes a disconnect
safe to resume with `--init-from`.

**Skip-training shortcut:** if `LOAD_FROM_DRIVE` is set in cell 3, this cell
mounts Drive, copies that file into `OUT_DIR/model.pt`, and skips training so
everything downstream (eval, reality probe) runs against the loaded checkpoint.


In [ ]:
from pathlib import Path
import shutil
from lines.train.train_autoregressive import ARTrainConfig, train_autoregressive

if LOAD_FROM_DRIVE is not None:
    # eval-only path: mount Drive (no-op if already mounted) and copy the
    # checkpoint into the same OUT_DIR that fresh training would have written.
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    src = Path(LOAD_FROM_DRIVE)
    if not src.exists():
        raise FileNotFoundError(f'LOAD_FROM_DRIVE points at a missing file: {src}')
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
    shutil.copy(src, Path(OUT_DIR) / 'model.pt')
    print(f'loaded {src} -> {OUT_DIR}/model.pt (training skipped)')
    result = None
else:
    cfg = ARTrainConfig(
        canvas_side=64,
        epochs=100,
        batch_size=64,
        d_model=256,
        n_decoder_layers=5,
        lr=3e-4,
        device='cuda',
        checkpoint_every=5,
    )
    result = train_autoregressive(
        cfg,
        train_dir=Path(TRAIN_DIR),
        test_dir=Path(TEST_DIR),
        out_dir=Path(OUT_DIR),
    )


## 6. Score under the honest F1 metric + per-image distribution

Mean F1 averages over edges and hides the **distribution** of per-image quality.
A 60% perfect / 40% catastrophic mix can still average F1=0.9, but only ~60% of
drawings would be *usable*. So we report two extras:

* `exact_match_rate` = fraction of held-out drawings reconstructed completely (F1 >= 0.99)
* `near_match_rate`  = fraction with at most ~1 wrong edge per 10 (F1 >= 0.9)


In [ ]:
import torch
from lines.datagen.dataset import Dataset
from lines.datagen.sampler2d import Canvas
from lines.eval.harness import run_predictor
from lines.models.autoregressive import AutoregressiveModel
from lines.train.predictor_ar import AutoregressivePredictor
from lines.baselines.classical import ClassicalBaseline

from pathlib import Path
ck = torch.load(Path(OUT_DIR) / 'model.pt', map_location='cuda', weights_only=False)
cfg = ck['cfg']
model = AutoregressiveModel(canvas_side=cfg['canvas_side'], d_model=cfg['d_model'],
                            n_heads=cfg['n_heads'], n_decoder_layers=cfg['n_decoder_layers'],
                            max_seq_len=cfg['max_seq_len']).cuda()
model.load_state_dict(ck['model'])

C = Canvas(cfg['canvas_side'], cfg['canvas_side'])
ar_greedy = AutoregressivePredictor(model, C, max_tokens=cfg['max_seq_len'],
                                     device='cuda', beam_size=1)

def _score(test_dir):
    ds = Dataset(test_dir)
    return ds, run_predictor(ar_greedy, ds, C)

if DATA_KIND == 'mixed':
    # Gate 4: EVAL ON BOTH PURE SPLITS SEPARATELY. averaging would hide a per-
    # content regression. boxes must hold >= 0.88, cylinders must hold >= 0.80.
    ds_box, r_box = _score('data/test64_box')
    ds_cyl, r_cyl = _score('data/test64_cyl')
    print('held-out content       F1       Exact   Near')
    print(f'boxes      (recorded)  0.980    0.932   0.932')
    print(f'cylinders  (recorded)  0.995    0.988   0.988')
    print(f'boxes      (this run)  {r_box["mean_f1"]:.3f}    {r_box["exact_match_rate"]:.3f}   {r_box["near_match_rate"]:.3f}')
    print(f'cylinders  (this run)  {r_cyl["mean_f1"]:.3f}    {r_cyl["exact_match_rate"]:.3f}   {r_cyl["near_match_rate"]:.3f}')
    print()
    box_ok = r_box['exact_match_rate'] >= 0.88
    cyl_ok = r_cyl['exact_match_rate'] >= 0.80
    if box_ok and cyl_ok:
        print(f'WIN  GATE 4 passed: boxes {r_box["exact_match_rate"]:.3f} >= 0.88 AND cylinders {r_cyl["exact_match_rate"]:.3f} >= 0.80. One model, both content types. Stage 2 SHIPS.')
    elif box_ok and not cyl_ok:
        print(f'CYL REGRESSED  cyl {r_cyl["exact_match_rate"]:.3f} < 0.80; cylinders did not survive mixing. Larger model or weighted sampling.')
    elif cyl_ok and not box_ok:
        print(f'BOX REGRESSED  box {r_box["exact_match_rate"]:.3f} < 0.88 (vs 0.932 pure). Catastrophic interference.')
    else:
        print(f'BOTH REGRESSED  box {r_box["exact_match_rate"]:.3f}, cyl {r_cyl["exact_match_rate"]:.3f}. Likely undertrained at this mix.')
    # use cylinder split for visualization (more interesting content)
    ar = ar_greedy; ds = ds_cyl; r_ar = r_cyl
else:
    ds, r_greedy = _score(TEST_DIR)
    r_base = run_predictor(ClassicalBaseline(), ds, C)
    print('Predictor                       F1       Exact   Near')
    print(f'classical baseline              {r_base["mean_f1"]:.3f}    {r_base["exact_match_rate"]:.3f}   {r_base["near_match_rate"]:.3f}')
    if DATA_KIND == 'boxes':
        print(f'set predictor + post-process    0.282    (recorded)')
        print(f'AR small preset (recorded)      0.899    0.642   0.650')
        print(f'AR BIG ({DATA_KIND}, this run)        {r_greedy["mean_f1"]:.3f}    {r_greedy["exact_match_rate"]:.3f}   {r_greedy["near_match_rate"]:.3f}')
    else:
        print(f'AR boxes (recorded ship)        0.980    0.932   0.932')
        print(f'AR BIG ({DATA_KIND}, this run)    {r_greedy["mean_f1"]:.3f}    {r_greedy["exact_match_rate"]:.3f}   {r_greedy["near_match_rate"]:.3f}')
    print()
    print(f'  perfect drawings: {r_greedy["n_perfect"]}/{r_greedy["n"]}'
          f'  ({r_greedy["exact_match_rate"]*100:.1f}%)')
    print(f'  near-perfect:     {r_greedy["n_near"]}/{r_greedy["n"]}'
          f'  ({r_greedy["near_match_rate"]*100:.1f}%)')
    ar = ar_greedy; r_ar = r_greedy
    exact_target = 0.85 if DATA_KIND == 'boxes' else 0.80
    if r_greedy['exact_match_rate'] >= exact_target:
        print(f'WIN  Exact {r_greedy["exact_match_rate"]:.3f} >= {exact_target}: {DATA_KIND} reconstruction ships.')
    elif r_greedy['exact_match_rate'] >= 0.60:
        print(f'PARTIAL  Exact {r_greedy["exact_match_rate"]:.3f}: still improving. Try +50 epochs (warm-start) or larger model.')
    else:
        print(f'PLATEAU  Exact {r_greedy["exact_match_rate"]:.3f}: capacity may not be the bottleneck. Inspect which views fail.')


## 6b. Beam search vs greedy on the *same* trained model

Greedy is myopic -- once it picks a bad token, the rest cascades (the row-3
catastrophe in the panel below). Beam-3 explores 3 paths and reduces that
rate. No retraining; just a different decoder. Slower than greedy on CPU but
comparable on GPU; expect ~+0.02 to ~+0.05 F1 and a bigger jump on
`exact_match_rate`.


In [ ]:
# Use r_ar (set by the previous cell) as the greedy reference. For mixed mode
# r_ar is the cylinder-split score; for pure modes it's the single test score.
ar_beam = AutoregressivePredictor(model, C, max_tokens=cfg['max_seq_len'],
                                   device='cuda', beam_size=3)
r_beam = run_predictor(ar_beam, ds, C)
print('Predictor                       F1       Exact   Near')
print(f'AR greedy (this run)            {r_ar["mean_f1"]:.3f}    {r_ar["exact_match_rate"]:.3f}   {r_ar["near_match_rate"]:.3f}')
print(f'AR beam-3 (this run)            {r_beam["mean_f1"]:.3f}    {r_beam["exact_match_rate"]:.3f}   {r_beam["near_match_rate"]:.3f}')
print()
delta_f1 = r_beam['mean_f1'] - r_ar['mean_f1']
delta_exact = r_beam['exact_match_rate'] - r_ar['exact_match_rate']
print(f'beam vs greedy: dF1 = {delta_f1:+.3f}  dExact = {delta_exact:+.3f}')
# use the better predictor for the visualization cell
if r_beam['mean_f1'] > r_ar['mean_f1']:
    ar = ar_beam
    print('-> using beam-3 for the visualization below')


## 7. Eyeball the wireframes

Six held-out boxes: input | model prediction | ground truth.


In [ ]:
import matplotlib.pyplot as plt
from lines.datagen.render import render_primitives

N = 6
fig, axes = plt.subplots(N, 3, figsize=(7, 2.2 * N))
for k in range(N):
    img, gt = ds[k]
    pred = ar(img)
    pred_img = render_primitives(pred, C.width, C.height)
    gt_img = render_primitives(gt, C.width, C.height)
    for ax, im, title in zip(axes[k], [img, pred_img, gt_img],
                              ['input', 'prediction', 'ground truth']):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title if k == 0 else ''); ax.axis('off')
plt.tight_layout(); plt.show()


## 7b. Reality check: perturbation probes

We've trained on a very specific distribution: orthographic, fixed view-jitter,
Pillow + supersampling at 1.5-3 px strokes, auto-fit to canvas. Any real input
differs along multiple axes. Each probe perturbs the held-out test data along
**one axis at a time** so we can see which shifts the model survives and which
break it.

* `baseline`       -- control (no perturbation)
* `cv2`            -- different rasterizer (OpenCV; different AA + endpoints)
* `jpeg`           -- JPEG roundtrip at q=60 (the project's inputs are .jpg)
* `stroke_thicker` -- 4.5 px (outside training 1.5-3 range)
* `stroke_thinner` -- 0.8 px (outside training)
* `off_center`     -- random translation (our generator always centers)
* `small_scale`    -- object only fills ~40% of canvas

Each probe runs ~200 samples through the trained model and reports F1 +
exact-match. The drop from `baseline` is the sim-to-real cost for that axis.


In [ ]:
import numpy as np
from lines.datagen.realityprobe import PERTURBATIONS, make_probe_sample
from lines.eval.metrics import evaluate

PROBE_N = 200
# probe the cylinder split if available (more interesting content); fall back to box
probe_test_dir = 'data/test64_cyl' if Path('data/test64_cyl/manifest.json').exists() else 'data/test64_box'
probe_ds = Dataset(probe_test_dir)
n_probe = min(PROBE_N, len(probe_ds))
print(f'probing against {probe_test_dir} (first {n_probe} samples)')
print()

rows = []
for name in ['baseline', 'cv2', 'jpeg', 'stroke_thicker', 'stroke_thinner',
             'off_center', 'small_scale']:
    f1s, exact = [], 0
    for i in range(n_probe):
        _, gt_base = probe_ds[i]
        img, gt = make_probe_sample(gt_base, C, name, seed=i * 31 + 1)
        pred = ar(img)
        m = evaluate(pred, gt, C)
        f1s.append(m['f1'])
        if m['f1'] >= 0.99:
            exact += 1
    mean_f1 = float(np.mean(f1s))
    exact_rate = exact / n_probe
    rows.append((name, mean_f1, exact_rate))

baseline_f1, baseline_exact = rows[0][1], rows[0][2]
print('perturbation         F1     Exact     dF1       dExact')
print('-' * 60)
for name, f1, ex in rows:
    df = f1 - baseline_f1
    de = ex - baseline_exact
    marker = '  <-- baseline' if name == 'baseline' else ''
    print(f'  {name:18s} {f1:.3f}  {ex:.3f}    {df:+.3f}    {de:+.3f}{marker}')
print()
worst_name = min(rows[1:], key=lambda r: r[1])[0]
best_name = max(rows[1:], key=lambda r: r[1])[0]
print(f'most robust axis:   {best_name}')
print(f'most fragile axis:  {worst_name}  <-- this is where to focus domain randomization')


## 7c. Worst-case visual: 6 samples from the most fragile probe


In [ ]:
import matplotlib.pyplot as plt
from lines.datagen.realityprobe import make_probe_sample
from lines.datagen.render import render_primitives
from lines.eval.metrics import evaluate

# pick 6 samples from the most fragile probe, sorted by WORST F1
samples = []
for i in range(min(60, len(probe_ds))):
    _, gt_base = probe_ds[i]
    img, gt = make_probe_sample(gt_base, C, worst_name, seed=i * 31 + 1)
    pred = ar(img)
    f1 = evaluate(pred, gt, C)['f1']
    samples.append((f1, img, pred, gt))
samples.sort(key=lambda s: s[0])
samples = samples[:6]

fig, axes = plt.subplots(6, 3, figsize=(7, 2.2 * 6))
for k, (f1, img, pred, gt) in enumerate(samples):
    pred_img = render_primitives(pred, C.width, C.height)
    gt_img = render_primitives(gt, C.width, C.height)
    for ax, im, title in zip(axes[k], [img, pred_img, gt_img],
                              [f'{worst_name} input  (F1={f1:.2f})', 'prediction', 'ground truth']):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title if k == 0 else (f'F1={f1:.2f}' if title.startswith(worst_name) else '')); ax.axis('off')
plt.tight_layout(); plt.show()


## 8. (Optional) Save the trained checkpoint to Google Drive

Colab sessions get reaped. Uncomment to persist the checkpoint.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/lines_checkpoints
# import shutil
# dst = f'/content/drive/MyDrive/lines_checkpoints/ar_{DATA_KIND}_64.pt'
# shutil.copy(f'{OUT_DIR}/model.pt', dst)
# print(f'saved to Drive: {dst}')
